In [ ]:
# Cell 1: Import Libraries
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_20newsgroups
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# For better visualization
%matplotlib inline
sns.set(style="whitegrid")


In [ ]:
# Cell 2: Load and Prepare Dataset
# Define the categories to use
categories = ['rec.autos', 'sci.space', 'talk.politics.misc', 'comp.graphics']

# Load the dataset
newsgroups = fetch_20newsgroups(subset='all', categories=categories, remove=('headers', 'footers', 'quotes'))

# Create a DataFrame
df = pd.DataFrame({
    'text': newsgroups.data,
    'target': newsgroups.target,
    'category': [newsgroups.target_names[i] for i in newsgroups.target]
})

# Display the first few rows
df.head()


In [ ]:
# Cell 3: Map Categories to Sentiments
# Define the sentiment mapping
sentiment_mapping = {
    'rec.autos': 'positive',
    'sci.space': 'positive',
    'talk.politics.misc': 'negative',
    'comp.graphics': 'neutral'
}

# Apply the mapping to create a new 'sentiment' column
df['sentiment'] = df['category'].map(sentiment_mapping)

# Check the distribution of sentiments
print("Sentiment Distribution:")
print(df['sentiment'].value_counts())


In [ ]:
# Cell 4: Split Dataset
# Features and labels
X = df['text']
y = df['sentiment']

# Split the dataset (80% training, 20% testing)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f'Training samples: {X_train.shape[0]}')
print(f'Testing samples: {X_test.shape[0]}')


In [ ]:
# Cell 5: Define Models and Train
# Define feature extraction methods
vectorizers = {
    'CountVectorizer': CountVectorizer(stop_words='english'),
    'TfidfVectorizer': TfidfVectorizer(stop_words='english')
}

# Define classifiers
classifiers = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'MultinomialNB': MultinomialNB(),
    'LinearSVC': LinearSVC(random_state=42)
}

# To store results
results = []


In [ ]:
# Train and evaluate each model
for vec_name, vectorizer in vectorizers.items():
    for clf_name, classifier in classifiers.items():
        # Create pipeline
        pipeline = Pipeline([
            ('vectorizer', vectorizer),
            ('classifier', classifier)
        ])

        # Train the model
        pipeline.fit(X_train, y_train)

        # Predict on test data
        y_pred = pipeline.predict(X_test)

        # Evaluate
        acc = accuracy_score(y_test, y_pred)
        report = classification_report(y_test, y_pred, output_dict=True)

        # Extract metrics for each sentiment
        precision_pos = report['positive']['precision']
        recall_pos = report['positive']['recall']
        f1_pos = report['positive']['f1-score']

        precision_neg = report['negative']['precision']
        recall_neg = report['negative']['recall']
        f1_neg = report['negative']['f1-score']

        precision_neu = report['neutral']['precision']
        recall_neu = report['neutral']['recall']
        f1_neu = report['neutral']['f1-score']

        # Store the results
        results.append({
            'Vectorizer': vec_name,
            'Classifier': clf_name,
            'Accuracy': acc,
            'Precision_Positive': precision_pos,
            'Recall_Positive': recall_pos,
            'F1_Positive': f1_pos,
            'Precision_Negative': precision_neg,
            'Recall_Negative': recall_neg,
            'F1_Negative': f1_neg,
            'Precision_Neutral': precision_neu,
            'Recall_Neutral': recall_neu,
            'F1_Neutral': f1_neu
        })

        print(f'Completed: {vec_name} + {clf_name} | Accuracy: {acc:.4f}')


In [ ]:
# Cell 6: Compare Models
# Create DataFrame
results_df = pd.DataFrame(results)

# Display the results table
results_df


In [ ]:
# Plot Accuracy
plt.figure(figsize=(12, 6))
sns.barplot(x='Vectorizer', y='Accuracy', hue='Classifier', data=results_df)
plt.title('Model Accuracy Comparison')
plt.ylim(0.6, 0.85)
plt.show()


In [ ]:
# Melt the DataFrame for easier plotting
f1_scores = results_df.melt(id_vars=['Vectorizer', 'Classifier'],
                            value_vars=['F1_Positive', 'F1_Negative', 'F1_Neutral'],
                            var_name='F1_Type',
                            value_name='F1_Score')

# Replace F1_Type values for better readability
f1_scores['F1_Type'] = f1_scores['F1_Type'].str.replace('F1_', '')

# Plot F1-Scores
plt.figure(figsize=(14, 7))
sns.barplot(x='Vectorizer', y='F1_Score', hue='Classifier', data=f1_scores)
plt.title('Model F1-Score Comparison Across Sentiments')
plt.ylim(0.6, 0.85)
plt.show()


In [ ]:
# Cell 7: Select Best Model
# For simplicity, we'll select the model with the highest Accuracy
best_model_idx = results_df['Accuracy'].idxmax()
best_model = results_df.loc[best_model_idx]

print("Best Model:")
print(best_model)


In [ ]:
# Cell 8: Final Model Training
# Define the best pipeline based on selected model
final_pipeline = Pipeline([
    ('vectorizer', TfidfVectorizer(stop_words='english')),
    ('classifier', LinearSVC(random_state=42))
])

# Train on the entire training set
final_pipeline.fit(X_train, y_train)

print("Final model trained successfully.")


In [ ]:
# Cell 9: Prediction Function
def predict_sentiment(text):
    prediction = final_pipeline.predict([text])[0]
    return prediction

# Example usage
examples = [
    "I love my new car, it's fantastic!",
    "The graphics card in my computer stopped working.",
    "I think the government's policies are flawed.",
    "Exploring the universe is fascinating.",
    "The movie was okay, not too bad but not great either.",
    "I ate the sandwich it has a bad taste",
    "I do not like this car, it has a bad color"
]

for sentence in examples:
    sentiment = predict_sentiment(sentence)
    print(f"Sentence: \"{sentence}\" | Sentiment: {sentiment}")
